# ORBINEX — Prithvi-EO-2.0-300M Authentic Cloud Pretrained Inference Pipeline
**Hardware Target:** Google Colab Free Tier (NVIDIA T4 GPU, 15.36 GB VRAM)
**Objective:** Pretrained forward pass verification of Change-UperNet on authentic Sentinel-2 6-band imagery.

### Strict Scientific Safeguards Active:
1. **Gate 1:** Official Checkpoint SHA-256 Checksum Verification (`faab2c8b776f0723...`)
2. **Gate 2:** Multispectral Manifest Integrity Check (24 scenes: 14 train / 10 test)
3. **Gate 3:** Physical 6-Band Completeness Check (`B02, B03, B04, B8A, B11, B12` on disk)
4. **Gate 4:** Temporal Bi-Epoch Validation ($T_1 < T_2$)
5. **Gate 5:** Ground-Truth Mask Alignment Check (exact dimension matching)
6. **Gate 6:** Zero Data Leakage (Disjoint train and test scenes)
7. **Gate 7:** Zero Heuristic Fallback (Strict `RuntimeError` on missing weights)
8. **Gate 8 (AI-08B4):** Real Sentinel-2 Pretrained Forward Pass Telemetry Verification

**CRITICAL:** AI-08C training remains strictly blocked. Cell 6 contains a hard execution halt.

In [ ]:
# Cell 1: Environment Setup & Hardware Telemetry
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q einops timm rasterio tifffile huggingface_hub pandas tqdm pillow

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("[HALT] CUDA GPU accelerator is not detected! In Colab, go to Runtime -> Change runtime type -> Select T4 GPU.")
print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
!nvidia-smi

In [ ]:
# Cell 2: Clone Repository & Setup Directory Tree
!git clone -b feat/satellite-foundation https://github.com/shashank/osiris-master.git
%cd osiris-master/osiris-master
!mkdir -p .cache/models/prithvi-eo-2.0-300m data/oscd_multispectral

import subprocess
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print(f"[REPO] Verified repository on branch feat/satellite-foundation at commit: {commit}")

In [ ]:
# Cell 3: GATE 1 — Download & Verify Official IBM-NASA Checkpoint
import os, hashlib, urllib.request

CHECKPOINT_URL = "https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M/resolve/main/Prithvi_EO_V2_300M.pt"
EXPECTED_SHA256 = "faab2c8b776f0723f93431747603a7ec586a54fc45473613b4322c29752486e7"
TARGET_PATH = ".cache/models/prithvi-eo-2.0-300m/Prithvi_EO_V2_300M.pt"

if not os.path.exists(TARGET_PATH):
    print("[GATE 1] Downloading official Prithvi-EO-2.0-300M weights from Hugging Face (~1.33 GB)... ")
    !wget -q --show-progress -O {TARGET_PATH} {CHECKPOINT_URL}

print("[GATE 1] Verifying SHA-256 Checksum...")
h = hashlib.sha256()
with open(TARGET_PATH, "rb") as f:
    while chunk := f.read(1024 * 1024 * 8):
        h.update(chunk)
actual_sha256 = h.hexdigest()

if actual_sha256.lower() != EXPECTED_SHA256.lower():
    os.remove(TARGET_PATH)
    raise RuntimeError(f"[GATE 1 FAILED] Checkpoint corrupted! Got {actual_sha256}, expected {EXPECTED_SHA256}")
print(f"[GATE 1 PASSED] Authentic IBM-NASA Checkpoint Verified: {actual_sha256}")

In [ ]:
# Cell 4: GATES 2 & 3 — Download Authentic OSCD Bands & Verify Physical Rasters
import os, sys, subprocess

# 1. Download and extract selective bands
res = subprocess.run([
    "python", "src/lib/satellite/prithvi_cloud/download_and_verify_oscd_multispectral.py",
    "--target_dir", "./data/oscd_multispectral",
    "--manifest", "src/lib/satellite/prithvi_cloud/oscd_multispectral_manifest.json"
])
if res.returncode != 0:
    raise RuntimeError("[GATE 2 FAILED] OSCD multispectral download and extraction failed!")

# 2. Run Programmatic Manifest Validator
res_val = subprocess.run(["python", "src/lib/satellite/prithvi_cloud/validate_multispectral_manifest.py"])
if res_val.returncode != 0:
    raise RuntimeError("[GATE 2 FAILED] Manifest validation failed!")

# 3. Physical Gate 3 Check: Assert all 6 GeoTIFF rasters exist for target scene 'bercy'
scene = "bercy"
required_bands = ["B02.tif", "B03.tif", "B04.tif", "B8A.tif", "B11.tif", "B12.tif"]
candidate_roots = [
    f"data/oscd_multispectral/Onera Satellite Change Detection dataset - Images/{scene}",
    f"data/oscd_multispectral/{scene}"
]
scene_base = next((r for r in candidate_roots if os.path.exists(r)), None)
if not scene_base:
    raise FileNotFoundError(f"[GATE 3 FAILED] Scene folder '{scene}' not found!")

for epoch in ["imgs_1_rect", "imgs_2_rect", "imgs_1", "imgs_2"]:
    epoch_dir = os.path.join(scene_base, epoch)
    if os.path.exists(epoch_dir):
        for b in required_bands:
            p = os.path.join(epoch_dir, b)
            if not os.path.exists(p):
                raise FileNotFoundError(f"[GATE 3 FAILED] Missing GeoTIFF band raster: {p}")

print(f"[GATE 3 PASSED] All 6 authentic multispectral GeoTIFF bands physically confirmed on disk for '{scene}'.")

In [ ]:
# Cell 5: GATES 4, 5, 6, 7 — Strict Pre-Flight Data & Model Assertions
import json, torch
from src.lib.satellite.prithvi_cloud.prithvi_model import PrithviChangeDetectionModel

manifest_file = "src/lib/satellite/prithvi_cloud/oscd_multispectral_manifest.json"
with open(manifest_file, "r") as f:
    manifest = json.load(f)

# Gate 6: Zero Data Leakage Assertion
train_scenes = [s["scene_id"] for s in manifest["scenes"] if s["canonical_split"] == "train"]
test_scenes = [s["scene_id"] for s in manifest["scenes"] if s["canonical_split"] == "test"]
overlap = set(train_scenes).intersection(set(test_scenes))
if overlap:
    raise RuntimeError(f"[GATE 6 FAILED] Severe Data Leakage detected: {overlap}")
print(f"[GATE 6 PASSED] Zero Data Leakage: 14 Train scenes strictly disjoint from 10 Test scenes.")

# Gate 7: Pretrained Model Loading Assertion (Zero Heuristic Fallback)
print("[GATE 7] Initializing PrithviChangeDetectionModel with official weights...")
model = PrithviChangeDetectionModel(checkpoint_path=TARGET_PATH, freeze_backbone=True)
print(f"[GATE 7 PASSED] Authentic 304.9M ViT-B backbone loaded. 6.2M Change-UperNet decoder ready.")

# Run Real Data Architecture Verification
!python src/lib/satellite/prithvi_cloud/verify_official_checkpoint_and_real_data.py

In [ ]:
# Cell 5B: GATE 8 (AI-08B4) — Authentic Pretrained Forward Pass on Real Multispectral Imagery
# Executes genuine forward pass using authentic Sentinel-2 pixels, extracting intermediate taps and telemetry.
# Strictly NO BACKPROPAGATION and NO LOSS COMPUTATION.
!python src/lib/satellite/prithvi_cloud/run_cloud_real_inference.py \
    --checkpoint_path .cache/models/prithvi-eo-2.0-300m/Prithvi_EO_V2_300M.pt \
    --data_dir ./data/oscd_multispectral \
    --scene bercy \
    --device cuda \
    --download_if_missing \
    --output_json real_inference_telemetry.json

# Display execution telemetry summary
import json
with open("real_inference_telemetry.json", "r") as f:
    telemetry = json.load(f)
print("\n=================== AUTHENTIC INFERENCE TELEMETRY ===================")
print(json.dumps(telemetry, indent=2))

In [ ]:
# Cell 6: HARD STOP — AI-08C TRAINING IS PROHIBITED
raise RuntimeError(
    "[HALT — AI-08C TRAINING STRICTLY PROHIBITED]\n"
    "Model fine-tuning and parameter updates remain blocked.\n"
    "Do not proceed past Cell 5B. Copy the printed JSON telemetry from Cell 5B into chat to verify AI-08B4."
)